<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/Qwen3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as f
import math

## RMS Norm

In [2]:
class RMSNormalization(nn.Module):
  def __init__(self, embed_dim, eps=1e-6):
    super().__init__()
    self.embed_dim=embed_dim
    self.gamma=nn.Parameter(torch.ones(embed_dim))
    self.eps=eps
  def norm(self, x):
    val= x.pow(2).mean(-1, keepdim=True)
    val=torch.sqrt(val+self.eps)
    return x/val
  def forward(self, x):
    return self.gamma*self.norm(x)

## KVCache

In [3]:
class KVCache(nn.Module):
  def __init__(self, num_layers):
    super().__init__()
    self.cache=[(None, None)]*num_layers

  def num_items(self):
    return sum(1 for k_cache, v_cache in self.cache if k_cache is not None)

  def add_item(self, key, value, layer_num):
    k_cache, v_cache=self.cache[layer_num]
    if k_cache is None:
      self.cache[layer_num]=(key, value)
    else:
      new_k_cache=torch.cat((k_cache, key), dim=1)
      new_v_cache=torch.cat((v_cache, value), dim=1)
      self.cache[layer_num]=(new_k_cache, new_v_cache)

  def get_item(self, layer_num):
    if not (0<=layer_num<len(self.cache)):
      raise ValueError(f"Layer index {layer_num} is out of range for a KV cache with {len(self.cache)} layers.")
    return self.cache[layer_num]

## Swish (for Siwglu)

In [4]:
class Swish(nn.Module):
  def __init__(self, beta=1):
    super().__init__()
    self.sigmoid=nn.Sigmoid()
    self.beta=beta
  def forward(self, x):
    return x*self.sigmoid(self.beta*x)

## FFN (With Swish as Swiglu)

In [5]:
class FeedForward(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.l1=nn.Linear(input_size, hidden_size, dtype=torch.float, bias=False)
    self.l2=nn.Linear(input_size, hidden_size,dtype=torch.float, bias=False)
    self.l3=nn.Linear(hidden_size, output_size, dtype=torch.float, bias=False)
    self.swish=Swish()
  def forward(self,x):
    x1=self.l1(x)
    x2=self.l2(x)
    swig_x=self.swish(x1)*x2 ##Swiglu
    return self.l3(swig_x)


## RoPE

In [6]:
class RoPE(nn.Module):
  def __init__(self,dim, base=10000.0, context_len=4096, batch_size=32):
    super().__init__()
    self.base=base
    self.context_len=context_len
    self.batch_size=batch_size
    self.dim=dim
    self.base=base

    assert self.dim%2==0, "Head dim must be divisible by 2"

    theta_num=torch.arange(0, self.dim,2).float()
    theta=1.0/(base**(theta_num/dim))
    self.register_buffer('theta', theta)

    positions=torch.arange(0, context_len, dtype=torch.float)
    angles=positions.unsqueeze(1)*self.theta.unsqueeze(0)

    self.register_buffer('cos_cached', torch.cos(angles))
    self.register_buffer('sin_cached', torch.sin(angles))

  def forward(self, x, start_pos=0):
    b,s,h,d=x.shape
    x=x.view(b,s,h,d//2, 2)
    cos=self.cos_cached[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)
    sin=self.sin_cached[start_pos:start_pos+s].unsqueeze(0).unsqueeze(2)
    x_rot=torch.stack([
        x[..., 0]*cos-x[..., 1]*sin,
        x[..., 0]*sin+x[..., 1]*cos
    ], dim=-1)
    return x_rot.view(b,s,h,d)



In [ ]:
x=torch.randn(size=(1,10,4,32))
rope=RoPE(32, context_len=10)
rope(x).shape

torch.Size([1, 10, 4, 32])

## GQA

In [7]:
def repeat_kv(x, nrep):
  b,s,h,d=x.shape
  if nrep==1:
    return x
  x=x[:,:,:,None,:].expand(b,s,h,nrep,d).reshape(b,s,h*nrep,d)
  return x

In [71]:
class GQA(nn.Module):
  def __init__(self, n_heads, base, embed_dims, head_dim, max_seq_len, max_batch_size, n_kv_heads=None, qk_norm=False, dropout=0.1, device='cpu'):
    super().__init__()
    self.q_heads=n_heads
    self.kv_heads=n_kv_heads if n_kv_heads is not None else n_heads
    self.embed_dims=embed_dims
    self.max_seq_len=max_seq_len
    self.max_batch_size=max_batch_size
    self.head_dim=head_dim
    self.kv_rep=self.q_heads//self.kv_heads
    self.device=device
    self.dropout=dropout
    self.dropout_layer=nn.Dropout(self.dropout)
    self.rope=RoPE(dim=self.head_dim, base=base, context_len=max_seq_len, batch_size=max_batch_size)

    self.Wq=nn.Linear(self.embed_dims, self.q_heads*self.head_dim, bias=False)
    self.Wk=nn.Linear(self.embed_dims, self.kv_heads*self.head_dim, bias=False)
    self.Wv=nn.Linear(self.embed_dims, self.kv_heads*self.head_dim, bias=False)
    self.out=nn.Linear(self.q_heads*self.head_dim, self.embed_dims, bias=False)
    self.qk_norm=qk_norm
    if self.qk_norm:
      self.q_norm=RMSNormalization(self.head_dim)
      self.k_norm=RMSNormalization(self.head_dim)
    else:
      self.qk_norm=self.k_norm=None

  def forward(self, x, start_pos, layer_idx, mask=None, kv_cache=None, inference=False):
    b,s,d=x.shape
    q=self.Wq(x)
    k=self.Wk(x)
    v=self.Wv(x)

    q=q.view(b,s,self.q_heads, self.head_dim)
    k=k.view(b,s,self.kv_heads, self.head_dim)
    v=v.view(b,s,self.kv_heads, self.head_dim)

    if self.qk_norm:
      q=self.q_norm(q)
      k=self.k_norm(k)

    xq=self.rope(q, start_pos)
    xk=self.rope(k, start_pos)

    if inference==True:
      kv_cache.add_item(xk, v, layer_idx)
      keys, values=kv_cache.get_item(layer_idx)

    else:
      keys=xk
      values=v
    kv_len=keys.shape[1]
    k=repeat_kv(keys, self.kv_rep)
    v=repeat_kv(values, self.kv_rep)

    q=xq.transpose(1,2)
    k=k.transpose(1,2)
    v=v.transpose(1,2)

    attn_weights=torch.matmul(q, k.transpose(2,3))/math.sqrt(self.head_dim)

    seq_q=q.shape[2]
    seq_k=k.shape[2]


    # if seq_q>1 or (inference==False):
    #   if inference==True and kv_cache is not None:
    #     mask=torch.zeros((seq_q, seq_k), dtype=torch.bool, device=x.device)
    #     for i in range(seq_q):
    #       current_abs_pos=start_pos+i
    #       if current_abs_pos+i<seq_k:
    #         mask[i, current_abs_pos+1:]=True
    #     attn_weights=attn_weights.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
    #   else:
    #     mask=torch.tril(torch.ones((seq_q, seq_k), dtype=torch.bool, device=x.device), diagonal=1)
    #     attn_weights=attn_weights.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))

    if seq_q > 1 or not inference:
            # Create causal mask
            mask = torch.triu(torch.ones(seq_q, seq_k, device=x.device, dtype=torch.bool), diagonal=1)
            attn_weights = attn_weights.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))

    attn_weights=f.softmax(attn_weights, dim=-1)
    attn_weights=attn_weights.type(torch.bfloat16) ##Change wrt model
    #attn_weights=self.dropout_layer(attn_weights)

    attn_output=torch.matmul(attn_weights, v) # B,n,s,hd
    attn_output= attn_output.transpose(1,2).contiguous()
    attn_output=attn_output.view(b,s,-1)
    attn_output=self.out(attn_output)
    return attn_output

## Transformer Block

In [72]:
class Transformer_block(nn.Module):
  def __init__(self, n_heads, base, embed_dims, head_dim, hidden_dim, max_seq_len, max_batch_size,qk_norm=False, n_kv_heads=None, dropout=0.1, device='cpu'):
    super().__init__()
    self.attn=GQA(n_heads=n_heads, base=base, embed_dims=embed_dims, head_dim=head_dim, max_seq_len=max_seq_len, max_batch_size=max_batch_size, n_kv_heads=n_kv_heads, qk_norm=qk_norm, dropout=dropout, device=device)
    self.ffn=FeedForward(embed_dims, hidden_dim, embed_dims)
    self.norm1=RMSNormalization(embed_dims)
    self.norm2=RMSNormalization(embed_dims)
    self.n_heads=n_heads
    self.embed_dims=embed_dims
    self.max_seq_len=max_seq_len
    self.max_batch_size=max_batch_size
    self.n_kv_heads=n_kv_heads
    self.dropout=dropout
    self.device=device

  def forward(self, x, start_pos, layer_idx, mask=None, kv_cache=None, inference=False):
    b,s,d=x.shape
    x_res=x
    x=self.norm1(x)
    x=self.attn(x, start_pos, layer_idx, mask, kv_cache, inference)
    x=x+x_res
    y_res=x
    y=self.norm2(x)
    y=self.ffn(y)
    y=y+y_res
    return y

## Qwen3 Model

In [73]:
class Qwen(nn.Module):
  def __init__(self,vocab_size, base, embed_dim,head_dim, hidden_dim, max_seq_len, max_batch_size, n_layers, n_heads, n_kv_heads=None, qk_norm=False,dropout=0.1,dtype=torch.bfloat16, device='cpu'):
    super().__init__()
    self.kv_cache=KVCache(num_layers=n_layers)
    self.embeddings=nn.Embedding(vocab_size, embed_dim)
    self.transform_layers=nn.ModuleList([
        Transformer_block(n_heads=n_heads,base=base, embed_dims=embed_dim,head_dim=head_dim, hidden_dim=hidden_dim, max_seq_len=max_seq_len,max_batch_size= max_batch_size, n_kv_heads=n_kv_heads, qk_norm=qk_norm, dropout=dropout, device=device)
        for _ in range(n_layers)
    ])
    self.norm=RMSNormalization(embed_dim)
    self.output_layer=nn.Linear(embed_dim, vocab_size, bias=False)

  def forward(self, x, start_pos,inference=False):
    b,s=x.shape
    x=self.embeddings(x)
    for i, layer in enumerate(self.transform_layers):
      x=layer(x, start_pos, i, kv_cache=self.kv_cache, inference=inference)
    x=self.norm(x)
    x=self.output_layer(x)
    return x


## Loading weights

In [45]:
QWEN3_CONFIG = {
        "vocab_size": 151_936,           # Vocabulary size
        "context_length": 40_960,        # Context length that was used to train the model
        "emb_dim": 1024,                 # Embedding dimension
        "n_heads": 16,                   # Number of attention heads
        "n_layers": 28,                  # Number of layers
        "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
        "head_dim": 128,                 # Size of the heads in GQA
        "qk_norm": True,                 # Whether to normalize queries and values in GQA
        "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
        "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
        "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
    }


In [74]:
vocab_size=151936
context_length=40960
embed_dim=1024
n_heads=16
n_layers=28
hidden_dim=3072
head_dim=128
qk_norm=True
n_kv_groups=8
rope_base=1000000
dtype=torch.bfloat16
device='cuda' if torch.cuda.is_available() else 'cpu'

In [75]:
model=Qwen(
    vocab_size=vocab_size,
    base=rope_base,
    embed_dim=embed_dim,
    head_dim=head_dim,
    hidden_dim=hidden_dim,
    max_seq_len=context_length,
    max_batch_size=1,
    n_layers=n_layers,
    n_heads=n_heads,
    n_kv_heads=n_kv_groups,
    qk_norm=qk_norm,
    dropout=0.1,
    dtype=dtype,
    device=device
)

In [76]:
model

Qwen(
  (kv_cache): KVCache()
  (embeddings): Embedding(151936, 1024)
  (transform_layers): ModuleList(
    (0-27): 28 x Transformer_block(
      (attn): GQA(
        (dropout_layer): Dropout(p=0.1, inplace=False)
        (rope): RoPE()
        (Wq): Linear(in_features=1024, out_features=2048, bias=False)
        (Wk): Linear(in_features=1024, out_features=1024, bias=False)
        (Wv): Linear(in_features=1024, out_features=1024, bias=False)
        (out): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNormalization()
        (k_norm): RMSNormalization()
      )
      (ffn): FeedForward(
        (l1): Linear(in_features=1024, out_features=3072, bias=False)
        (l2): Linear(in_features=1024, out_features=3072, bias=False)
        (l3): Linear(in_features=3072, out_features=1024, bias=False)
        (swish): Swish(
          (sigmoid): Sigmoid()
        )
      )
      (norm1): RMSNormalization()
      (norm2): RMSNormalization()
    )
  )
  (norm): RMS

## Total params

In [49]:
def model_mem_size(model, ip=torch.float32):
  total_params=0
  total_grads=0

  for param in model.parameters():
    param_size=param.numel()
    total_params+=param_size
    if param.requires_grad:
      total_grads+=param_size

  total_buffers=sum(buf.numel() for buf in model.buffers())
  element_size=torch.tensor(0, dtype=ip).element_size()
  total_mem=(total_params+total_grads+total_buffers)*element_size
  return total_mem/(1024**3)

In [77]:
model_mem_size(model, torch.bfloat16)

3.0734896659851074

## Weight Loading

In [52]:
def load_weights_into_qwen(model, params, n_layers):
  def assign(left, right, tensor_name='unknown'):
    if left.shape!=right.shape:
      raise ValueError(f"Shape mismatch between tensor {tensor_name}. left shape: {left.shape}, right_shape: {right.shape}")
    return nn.Parameter(right.clone().detach() if isinstance(right, torch.Tensor) else torch.tensor(right))

  model.embeddings.weight=assign(model.embeddings.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

  for l in range(n_layers):
    block=model.transform_layers[l]
    attn=block.attn

    attn.Wq.weight=assign(
        attn.Wq.weight,
        params[f"model.layers.{l}.self_attn.q_proj.weight"],
        f"model.layers.{l}.self_attn.q_proj.weight"
    )
    attn.Wk.weight=assign(
        attn.Wk.weight,
        params[f"model.layers.{l}.self_attn.k_proj.weight"],
        f"model.layers.{l}.self_attn.k_proj.weight"
    )
    attn.Wv.weight=assign(
        attn.Wv.weight,
        params[f"model.layers.{l}.self_attn.v_proj.weight"],
        f"model.layers.{l}.self_attn.v_proj.weight"
    )
    attn.out.weight=assign(
        attn.out.weight,
        params[f"model.layers.{l}.self_attn.o_proj.weight"],
        f"model.layers.{l}.self_attn.o_proj.weight"
    )

    #Qk norms
    if hasattr(attn, 'q_norm') and attn.q_norm is not None:
      attn.q_norm.gamma=assign(
          attn.q_norm.gamma,
          params[f"model.layers.{l}.self_attn.q_norm.weight"],
          f"model.layers.{l}.self_attn.q_norm.weight"
      )
      attn.k_norm.gamma=assign(
          attn.k_norm.gamma,
          params[f"model.layers.{l}.self_attn.k_norm.weight"],
          f"model.layers.{l}.self_attn.k_norm.weight"
      )
    block.norm1.gamma=assign(
        block.norm1.gamma,
        params[f"model.layers.{l}.input_layernorm.weight"],
        f"model.layers.{l}.input_layernorm.weight"
    )
    block.norm2.gamma=assign(
        block.norm2.gamma,
        params[f"model.layers.{l}.post_attention_layernorm.weight"],
        f"model.layers.{l}.post_attention_layernorm.weight"
    )
    block.ffn.l1.weight=assign(
        block.ffn.l1.weight,
        params[f"model.layers.{l}.mlp.gate_proj.weight"],
        f"model.layers.{l}.mlp.gate_proj.weight"
    )
    block.ffn.l2.weight=assign(
        block.ffn.l2.weight,
        params[f"model.layers.{l}.mlp.up_proj.weight"],
        f"model.layers.{l}.mlp.up_proj.weight"
    )
    block.ffn.l3.weight=assign(
        block.ffn.l3.weight,
        params[f"model.layers.{l}.mlp.down_proj.weight"],
        f"model.layers.{l}.mlp.down_proj.weight"
    )

    model.norm.gamma=assign(
        model.norm.gamma,
        params["model.norm.weight"],
        "model.norm.weight"
    )
    if "lm_head.weight" in params:
      model.output_layer.weight=assign(
          model.output_layer.weight,
          params["lm_head.weight"],
          "lm_head.weight"
      )
    else:
        print("Model uses weight tying")
        model.output_layer.weight=assign(
            model.output_layer.weight,
            params["model.embed_tokens.weight"],
            "model.embed_tokens.weight"
        )


In [78]:
import json
from huggingface_hub import hf_hub_download, snapshot_download
from pathlib import Path
from safetensors.torch import load_file

repo_id=f"Qwen/Qwen3-0.6B-Base"

local_dir=Path(repo_id).parts[-1]
weight_file=hf_hub_download(
    repo_id,
    filename="model.safetensors",
    local_dir=local_dir,
)
weights_dict=load_file(weight_file)

load_weights_into_qwen(model, weights_dict, n_layers=n_layers)


Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying
Model uses weight tying


## Top-P sampling

In [20]:
def top_p_sampling(logits, p):
  probs_sort, prob_idx=torch.sort(logits, -1, descending=True)
  probs_sum=torch.cumsum(probs_sort, dim=-1)
  mask=probs_sum-probs_sort>p
  probs_sort[mask]=0
  probs_sort.div_(probs_sort.sum(dim=-1, keepdim=True))
  next_token=torch.multinomial(probs_sort, num_samples=1)
  next_token=torch.gather(prob_idx, -1, next_token)
  return next_token

## Inference

In [21]:
def inference(model, tokenizer, prompt, max_tokens_to_generate, temperature, top_p, stop_token, device):
  prompt_tokens=tokenizer.encode(prompt)
  model.to(device)
  model.kv_cache = KVCache(num_layers=len(model.transform_layers))
  generated_ids=torch.tensor([prompt_tokens], device=device) # Initialize as a tensor
  prompt_tokens = torch.tensor([prompt_tokens], device=device) # Convert list to tensor
  b,s=prompt_tokens.shape
  if not stop_token:
        stop_tokens = [tokenizer.eos_token_id]
        if tokenizer._special_to_id.get("<|im_end|>"):
            stop_tokens.append(tokenizer._special_to_id["<|im_end|>"])
  else:
    stop_tokens = stop_token


  with torch.no_grad():
    start_pos=0
    logits=model(prompt_tokens, start_pos, inference=True) # Use prompt_tokens (tensor) here
    for step in range(max_tokens_to_generate):
      next_token_log=logits[:, -1, :]/temperature
      next_token_log=torch.softmax(next_token_log, dim=-1)
      next_token=top_p_sampling(next_token_log, top_p)
      if next_token.item() in stop_tokens: # Use stop_tokens here
        break
      generated_ids=torch.cat([generated_ids, next_token], dim=-1)
      start_pos=start_pos+1
      logits=model(generated_ids, start_pos, inference=True) # Pass generated_ids (tensor) directly

  return tokenizer.decode(generated_ids.squeeze(0).tolist()) # Convert back to list for decoding

## Tokenizer

In [66]:

import re
import torch
from pathlib import Path
from tokenizers import Tokenizer


class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {t: self._tok.token_to_id(t) for t in self._SPECIALS}

        self.pad_token_id = self._special_to_id.get("<|endoftext|>")
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)
        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s


In [68]:
tokenizer_path=f"Qwen3-{0.6}B-Base/tokenizer.json"
repo_id=f"Qwen/Qwen3-0.6B-Base"

# Download the tokenizer file
tokenizer_file = hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=f"Qwen3-{0.6}B-Base",
)

tokenizer=Qwen3Tokenizer(tokenizer_file_path=tokenizer_file, repo_id=repo_id)

In [69]:
prompt="What is an LLM"
max_tokena=100
temperature=0.7
p=0.9

In [25]:
def greedy_inference(
    model,
    tokenizer,
    prompt,
    max_tokens_to_generate,
    temperature,
    top_p,
    stop_token,
    device
):
  model.eval()
  with torch.no_grad():
    prompt_tokens=tokenizer.encode(prompt)
    prompt_tokens=torch.tensor([prompt_tokens], device=device)
    b,s=prompt_tokens.shape
    for _ in range(max_tokens_to_generate):
      logits=model(prompt_tokens, 0, inference=True)
      next_token_log=logits[:, -1, :]/temperature
      next_token=torch.argmax(next_token_log, dim=-1)
      prompt_tokens=torch.cat([prompt_tokens, next_token.unsqueeze(0)], dim=-1) # Add unsqueeze here
      if next_token.item() in stop_token:
        break
  return tokenizer.decode(prompt_tokens.squeeze(0).tolist())

In [26]:
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

'<|im_start|>user\nGive me a short introduction to large language models.<|im_end|>\n'

In [89]:
def inference_fixed(model, tokenizer, prompt, max_tokens_to_generate, temperature, top_p, stop_token, device):
    # Don't apply chat template for base models
    prompt_tokens = tokenizer.encode(prompt, chat_wrapped=False)
    model.to(device)

    # Reset KV cache
    model.kv_cache = KVCache(num_layers=len(model.transform_layers))

    # Convert to tensor
    input_ids = torch.tensor([prompt_tokens], device=device)
    generated_ids = input_ids.clone()

    if not stop_token:
        stop_tokens = [tokenizer.eos_token_id]
    else:
        stop_tokens = stop_token

    model.eval()
    with torch.no_grad():
        # Process initial prompt
        start_pos = 0
        logits = model(input_ids, start_pos, inference=True)
        start_pos = input_ids.shape[1]  # Update start_pos to end of prompt

        # Generate tokens one by one
        for step in range(max_tokens_to_generate):
            # Get logits for last token
            next_token_logits = logits[:, -1, :] / temperature
            next_token_probs = torch.softmax(next_token_logits, dim=-1)

            # Sample next token
            if top_p < 1.0:
                next_token = top_p_sampling(next_token_probs, top_p)
            else:
                next_token = torch.argmax(next_token_probs, dim=-1, keepdim=True)

            # Check for stop tokens
            if next_token.item() in stop_tokens:
                break

            # Add to generated sequence
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)

            # Get logits for next iteration (pass only the new token)
            logits = model(next_token, start_pos, inference=True)
            start_pos += 1

    return tokenizer.decode(generated_ids.squeeze(0).tolist())


def correct_incremental_inference(model, tokenizer, prompt, max_tokens_to_generate, temperature, top_p, stop_token, device):
    """Correctly implements incremental KV caching - only new tokens after first pass"""

    prompt_tokens = tokenizer.encode(prompt, chat_wrapped=False)
    model.to(device)
    model.eval()

    # Reset KV cache
    model.kv_cache = KVCache(num_layers=len(model.transform_layers))

    # Convert to tensor
    prompt_tensor = torch.tensor([prompt_tokens], device=device)
    generated_tokens = []

    if not stop_token:
        stop_tokens = [tokenizer.eos_token_id]
    else:
        stop_tokens = stop_token

    with torch.no_grad():
        # FIRST PASS: Process entire prompt to populate KV cache
        start_pos = 0
        logits = model(prompt_tensor, start_pos, inference=True)
        current_pos = prompt_tensor.shape[1]  # Position after prompt

        # GENERATION LOOP: Process only new tokens
        for step in range(max_tokens_to_generate):
            # Sample next token from current logits
            next_token_logits = logits[:, -1, :] / temperature
            next_token_probs = torch.softmax(next_token_logits, dim=-1)

            if top_p < 1.0:
                next_token = top_p_sampling(next_token_probs, top_p)
            else:
                next_token = torch.argmax(next_token_probs, dim=-1, keepdim=True)

            # Check for stop tokens
            if next_token.item() in stop_tokens:
                break

            generated_tokens.append(next_token.item())

            # For next iteration: process ONLY the new token
            # KV cache already contains all previous context
            if step < max_tokens_to_generate - 1:
                logits = model(next_token, current_pos, inference=True)
                current_pos += 1

            if len(generated_tokens) > 1000:  # Safety check
                break

    # Combine prompt and generated tokens
    all_tokens = prompt_tokens + generated_tokens
    return tokenizer.decode(all_tokens)


# Let's also debug what's happening in your KV cache
def debug_kv_cache_inference(model, tokenizer, prompt, max_tokens_to_generate, device):
    """Debug version to see what's happening with KV cache"""

    prompt_tokens = tokenizer.encode(prompt, chat_wrapped=False)
    model.to(device)
    model.eval()

    # Reset KV cache
    model.kv_cache = KVCache(num_layers=len(model.transform_layers))

    prompt_tensor = torch.tensor([prompt_tokens], device=device)
    generated_tokens = []

    print(f"Prompt tokens: {len(prompt_tokens)}")
    print(f"Prompt: '{tokenizer.decode(prompt_tokens)}'")

    with torch.no_grad():
        # Process prompt
        print("\n=== Processing Prompt ===")
        logits = model(prompt_tensor, 0, inference=True)

        # Check KV cache after prompt
        k_cache, v_cache = model.kv_cache.get_item(0)  # Check first layer
        print(f"KV cache shape after prompt: {k_cache.shape if k_cache is not None else None}")

        current_pos = prompt_tensor.shape[1]

        # Generate a few tokens
        for step in range(min(5, max_tokens_to_generate)):
            print(f"\n=== Generation Step {step + 1} ===")

            # Sample token
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            print(f"Generated token: {next_token.item()} ('{tokenizer.decode([next_token.item()])}')")

            if next_token.item() == tokenizer.eos_token_id:
                break

            generated_tokens.append(next_token.item())

            # Process only new token
            if step < 4:  # Don't compute on last iteration
                logits = model(next_token, current_pos, inference=True)
                current_pos += 1

                # Check KV cache growth
                k_cache, v_cache = model.kv_cache.get_item(0)
                print(f"KV cache shape after step {step + 1}: {k_cache.shape}")

    all_tokens = prompt_tokens + generated_tokens
    result = tokenizer.decode(all_tokens)
    print(f"\nFinal result: '{result}'")
    return result
    """True incremental inference - only processes new tokens after the first pass"""

    prompt_tokens = tokenizer.encode(prompt, chat_wrapped=False)
    model.to(device)
    model.eval()

    # Reset KV cache
    model.kv_cache = KVCache(num_layers=len(model.transform_layers))

    # Convert to tensor
    input_ids = torch.tensor([prompt_tokens], device=device)

    if not stop_token:
        stop_tokens = [tokenizer.eos_token_id]
    else:
        stop_tokens = stop_token

    with torch.no_grad():
        # First pass: process the entire prompt
        start_pos = 0
        logits = model(input_ids, start_pos, inference=True)
        current_pos = input_ids.shape[1]

        # Generation loop: process one token at a time
        for step in range(max_tokens_to_generate):
            # Get next token from current logits
            next_token_logits = logits[:, -1, :] / temperature
            next_token_probs = torch.softmax(next_token_logits, dim=-1)

            # Sample next token
            if top_p < 1.0:
                next_token = top_p_sampling(next_token_probs, top_p)
            else:
                next_token = torch.argmax(next_token_probs, dim=-1, keepdim=True)

            # Check for stop tokens
            if next_token.item() in stop_tokens:
                break

            # Add to sequence
            input_ids = torch.cat([input_ids, next_token], dim=-1)

            # Process only the new token for next iteration
            if step < max_tokens_to_generate - 1:  # Don't compute logits on last iteration
                logits = model(next_token, current_pos, inference=True)
                current_pos += 1

            # Safety check
            if input_ids.shape[1] > 2048:
                break

    return tokenizer.decode(input_ids.squeeze(0).tolist())


# Test the fixed inference
if __name__ == "__main__":
    prompt = "Give me a short introduction to large language models."
    # Try the fixed KV cache version
    print("Fixed KV Cache Inference:")
    result2 = inference_fixed(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_tokens_to_generate=50,
        temperature=0.9,
        top_p=0.9,
        stop_token=[tokenizer.eos_token_id],
        device=device
    )
    print(result2)

Fixed KV Cache Inference:
Give me a short introduction to large language models. Large language models are general introduction. AI tools, might help me to learn python programming. Explain to Large language Models,. you know, I large language Model. can I give you a short introduction about Large Language:. Large Language Models are generally
